## 00 - Setup

### Configura Paths

In [1]:
from pathlib import Path
import getpass
import os

USER_NAME = getpass.getuser()
PATH_HOME = Path.home()
PROJECT_ROOT_PATH = PATH_HOME / "projects/brazilian_address_linkage"
DATALAKE_PATH = Path(f"/media/{USER_NAME}/Seagate Portable Drive/Datalake/")
MODELO_SPLINK_PATH = PROJECT_ROOT_PATH / "splink_models/cnefe_dedup_model.json"

os.chdir(PROJECT_ROOT_PATH)

## 01 -  Set Sessão spark

In [2]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from splink.backends.spark import similarity_jar_location
from splink import SparkAPI
from pathlib import Path
import warnings
import os

warnings.filterwarnings("ignore")

UF = "RJ"

conf = SparkConf()
conf.set("spark.driver.memory", "15g")
conf.set("spark.master", "local[4]")
conf.set("spark.local.dir", "spark_tmp/")
conf.set("spark.sql.shuffle.partitions", "600")
conf.set("spark.shuffle.checksum.enabled", "true")
conf.set("spark.shuffle.detectCorrupt", "true")
path = similarity_jar_location()
conf.set("spark.jars", path)

sc = SparkContext.getOrCreate(conf=conf)

spark = SparkSession(sc)
spark.sparkContext.setCheckpointDir("./tmp_checkpoints")
db_api = SparkAPI(spark_session=spark)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/08 21:55:33 WARN Utils: Your hostname, lucas-vital-Q570M-D3H, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp8s0)
26/10/08 21:55:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/10/08 21:55:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 21:55:33 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).
26/10/08 21:55:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 03 - Le dados limpos

In [3]:
cnefe_df = spark.read.parquet(
    os.path.join(DATALAKE_PATH, "silver/cnefe/cleaned_cnefe_addresses.parquet")
)

## 04 - Carrega Modelo

In [4]:
from splink import Linker

linker = Linker(cnefe_df, MODELO_SPLINK_PATH, db_api=db_api,
                input_table_aliases=["cnefe"])

26/10/08 21:55:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
SETTINGS VALIDATION: Errors were identified in your settings dictionary. 

Setting: `unique_id_column_name`

       - Missing column(s) from input dataframe(s): ``unique_id``

You may want to verify your settings dictionary has valid inputs in all fields before continuing.


## 05 -  Realiza predições

In [5]:
df_predict = linker.inference.predict(
    # [ESCALA] ERA 0.001. Filtrar cedo corta a saida em cerca de uma ordem de
    # grandeza. A cauda abaixo de 0,5 quase nao tem vinculos verdadeiros — o
    # caderno 03 mostrou que pares com um unico sinal forte nunca sao vinculo.
    threshold_match_probability=0.5,
    # [SPARK] materializa a tabela de nos com as frequencias de termos em vez de
    # recalcula-la dentro de um CTE gigante
    materialise_after_computing_term_frequencies=True,
    # [SPARK] materializa os pares bloqueados: ajuda o planner quando ha blocos
    # grandes e evita replanejar o blocking a cada estagio
    materialise_blocked_pairs=True,
)

# [SPARK] as_spark_dataframe() em vez de as_pandas_dataframe(): continua distribuido
pares = df_predict.as_spark_dataframe()
print(f"Colunas na saida: {len(pares.columns)}")

26/10/08 23:28:07 WARN Utils: Suppressing exception in finally: No space left on device
java.io.IOException: No space left on device
	at java.base/java.io.FileOutputStream.writeBytes(Native Method)
	at java.base/java.io.FileOutputStream.write(FileOutputStream.java:349)
	at org.apache.spark.storage.TimeTrackingOutputStream.write(TimeTrackingOutputStream.java:59)
	at org.apache.spark.io.MutableCheckedOutputStream.write(MutableCheckedOutputStream.scala:43)
	at java.base/java.io.BufferedOutputStream.flushBuffer(BufferedOutputStream.java:81)
	at java.base/java.io.BufferedOutputStream.write(BufferedOutputStream.java:127)
	at net.jpountz.lz4.LZ4BlockOutputStream.flushBufferedData(LZ4BlockOutputStream.java:225)
	at net.jpountz.lz4.LZ4BlockOutputStream.finish(LZ4BlockOutputStream.java:257)
	at net.jpountz.lz4.LZ4BlockOutputStream.close(LZ4BlockOutputStream.java:195)
	at org.apache.spark.storage.DiskBlockObjectWriter.$anonfun$closeResources$2(DiskBlockObjectWriter.scala:190)
	at org.apache.spark

Py4JJavaError: An error occurred while calling o96.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 234 in stage 63.0 failed 1 times, most recent failure: Lost task 234.0 in stage 63.0 (TID 12067) (192.168.100.25 executor driver): java.io.IOException: No space left on device
	at java.base/java.io.FileOutputStream.writeBytes(Native Method)
	at java.base/java.io.FileOutputStream.write(FileOutputStream.java:349)
	at org.apache.spark.storage.TimeTrackingOutputStream.write(TimeTrackingOutputStream.java:59)
	at org.apache.spark.io.MutableCheckedOutputStream.write(MutableCheckedOutputStream.scala:43)
	at java.base/java.io.BufferedOutputStream.flushBuffer(BufferedOutputStream.java:81)
	at java.base/java.io.BufferedOutputStream.write(BufferedOutputStream.java:127)
	at net.jpountz.lz4.LZ4BlockOutputStream.flushBufferedData(LZ4BlockOutputStream.java:225)
	at net.jpountz.lz4.LZ4BlockOutputStream.write(LZ4BlockOutputStream.java:178)
	at org.apache.spark.storage.Dis

In [ ]:
pares.write.parquet("data/rio_de_janeiro_pairs")